In [ ]:
import os
import sys


def _ensure_repo_root_on_path() -> None:
    current = os.getcwd()
    while True:
        if os.path.isdir(os.path.join(current, "examples")):
            if current not in sys.path:
                sys.path.insert(0, current)
            break
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent


_ensure_repo_root_on_path()

from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage
from pydantic import SecretStr
from examples.read_env import read_env
from examples.action_api_samples.api_sample import (
    load_adopt_profile
)
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from examples.action_api_samples.adopt_client import AdoptClient
from capabilities_registry.capability_registry import CapabilityRegistry
from langgraph.checkpoint.memory import InMemorySaver 
from examples.tool_calling_samples.hitl.helpers import draw_mermaid_png


In [ ]:
profile = load_adopt_profile()
adopt_client = AdoptClient()
registry = CapabilityRegistry(adopt_client)
adopt_env = read_env()
print("✓ Loaded adopt profile")
print("✓ Loaded client")
print("✓ Loaded environment configuration")

In [ ]:
tools = await registry.get_tools_for_keys(
            capability_keys=[
                "create_new_project",
                "list_all_projects",
                "get_project_activity_feed",
                "search_project_by_name",
                "manage_user_profiles",
                "update_user_preferences",
                "get_team_members_list",
                "invite_user_to_team",
                "remove_user_from_team",
                "get_system_health_status"
            ],
            profile=profile,
            execution_type="TOOL"
        )
print("\n🔧 Creating LangChain tools...")


In [ ]:
model = ChatBedrockConverse(
        model=adopt_env.BEDROCK_MODEL,
        region_name=adopt_env.AWS_REGION,
        aws_access_key_id=SecretStr(adopt_env.AWS_ACCESS_KEY_ID),
        aws_secret_access_key=SecretStr(adopt_env.AWS_SECRET_ACCESS_KEY),
    )
agent = create_agent(
        model=model,
        tools=tools,
        middleware=[
        HumanInTheLoopMiddleware(
                interrupt_on={
                    "create_new_project": {
                        "allowed_decisions": ["approve", "reject", "edit"]
                    }
                }
            )
        ],
        checkpointer=InMemorySaver(),
        name="projects_manager_agent",
        system_prompt="You are an autonomous project Management Agent specialized in B2B marketing project intelligence and optimization. Your primary responsibility is to continuously monitor project performance, recommend strategic optimizations, and automate project lifecycle management with minimal manual intervention. You help marketers stay ahead of search trends and engagement patterns while ensuring projects are always tuned for maximum performance and pipeline generation",
    )

draw_mermaid_png(agent)